# Estudo de Casos Propostos no Problema das Marginais Quânticas (QMP)

Este notebook implementa três exemplos benchmark clássicos para ilustrar os diferentes regimes de representabilidade do **Problema das Marginais Quânticas (QMP)** em sistemas de 3 qubits ($A, B, C$ com dimensão local $d=2$):

1. **Exemplo 1 (Marginais do Estado GHZ)**: Marginais mistas clássicas compatíveis com extensão pura e mista.
2. **Exemplo 2 (Marginais Maximamente Mistas)**: Marginais $I_4/4$ compatíveis com extensão mista (estado $I_8/8$), mas incompatíveis com qualquer estado puro.
3. **Exemplo 3 (Incompatibilidade de Bell)**: Marginais de Bell $|\Phi^+\rangle\langle\Phi^+|$ incompatíveis devido à Monogamia do Emaranhamento.


## 1. Importação de Bibliotecas e Módulos Fundamentais

Carregamento das funções primitivas (`funcoes_fundamentais.py`), manipuladores numéricos e bibliotecas de otimização.


In [1]:
from funcoes_fundamentais import *
import pandas as pd
import numpy as np


### 1.1 Instalação Automatizada das Dependências

Instalação dos solvers de otimização convexa e pacotes quânticos (`spacecore==0.4.2`, `sdplab`, `clarabel`, `cvxpy`).


In [ ]:
# Instalação das dependências necessárias no ambiente
# Nota: spacecore==0.4.2 é recomendado para total compatibilidade com sdplab==0.0.1
! pip install -q numpy matplotlib scipy "spacecore==0.4.2" sdplab cvxpy clarabel scs jax

### 1.2 Configuração de Versões e Importações Específicas

Verificação das versões instaladas dos pacotes `SDPLab` e `SpaceCore` e configuração da precisão de exibição do `numpy`.


In [3]:
from __future__ import annotations

import sys
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
# Bibliotecas SpaceCore e SDPLab para Programação Semidefinida 
import spacecore
from spacecore import Context, DenseVectorSpace, HermitianSpace, NumpyOps
import sdplab
from sdplab import DenseConstraintOp, SDPProblem
from sdplab.solvers import run_cvxpy_solver

np.set_printoptions(precision=4, suppress=True)

from funcoes_fundamentais import zero, one, ketbra, verifica_densidade, verify_marginals, base_hermitiana, ghz, partial_trace, tensor, embed_operator, solve_qmp, build_qmp_sdp, parse_subsystem_key, basis
print(f"Versão SDPLab:    {sdplab.__version__}")
print(f"Versão SpaceCore: {spacecore.__version__}")

Versão SDPLab:    0.0.1
Versão SpaceCore: 0.4.2


## 2. Exemplo 1: Marginais do Estado GHZ

### Descrição Física:
Para o estado puro de 3 qubits $|GHZ\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}$, a eliminação de qualquer um dos qubits via traço parcial resulta na mesma mistura clássica de 2 qubits:

$$\omega_{AB} = \omega_{AC} = \omega_{BC} = \frac{1}{2}(|00\rangle\langle 00| + |11\rangle\langle 11|) = \begin{pmatrix} 0.5 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0.5 \end{pmatrix}$$

### Regime de Representabilidade Esperado:
- **Representabilidade Mista**: **FACTÍVEL**. O SDP encontra uma extensão global válida $\rho_{ABC} \ge 0$.
- **Representabilidade Pura**: **FACTÍVEL**, pois o estado puro $|GHZ\rangle$ é a origem de tais marginais.


In [3]:
# Montagem das marginais do estado GHZ
omega_ghz_2qubits = 0.5 * (ketbra(np.kron(zero, zero)) + ketbra(np.kron(one, one)))

marginals_ex1 = {
    "AB": omega_ghz_2qubits,
    "AC": omega_ghz_2qubits,
    "BC": omega_ghz_2qubits,
}

print("Exemplo 1 (GHZ)")
sdp_ex1, M_ex1, b_ex1 = build_qmp_sdp(marginals_ex1, dims=[2, 2, 2])
print(f"SDP construído: dimensão do domínio = {sdp_ex1.dom.n}x{sdp_ex1.dom.n}, restrições m = {len(b_ex1)}")

# Resolução com CLARABEL
res_ex1 = solve_qmp(sdp_ex1, solver="CLARABEL")
print(f"Status do Solver: {res_ex1['status']} | Factível: {res_ex1['is_feasible']}")

# Auditoria
verif_ex1 = verify_marginals(res_ex1["rho"], marginals_ex1, dims=[2, 2, 2])
print(f"Erro de Hermiticidade:  {verif_ex1['hermitian_error']:.2e}")
print(f"Traço Tr(rho):          {verif_ex1['trace'].real:.12f} (Erro: {verif_ex1['trace_error']:.2e})")
print(f"lambda_min:             {verif_ex1['min_eigenvalue']:.2e} -> rho >= 0: {verif_ex1['is_psd']}")
print(f"Pureza Tr(rho^2):       {verif_ex1['purity']:.4f}")
print("Autovalores de rho_ABC:")
print(" ", np.round(verif_ex1["eigenvalues"], 6))
print("Resíduos por marginal:")
for k, err in verif_ex1["marginal_errors"].items():
    print(f"  Marginal {k}: Erro Frobenius = {err['frobenius_error']:.2e}, Max Abs = {err['max_abs_error']:.2e}")

Exemplo 1 (GHZ)
SDP construído: dimensão do domínio = 8x8, restrições m = 49
Status do Solver: optimal | Factível: True
Erro de Hermiticidade:  0.00e+00
Traço Tr(rho):          1.000000000000 (Erro: 2.44e-15)
lambda_min:             1.23e-15 -> rho >= 0: True
Pureza Tr(rho^2):       0.5000
Autovalores de rho_ABC:
  [0.  0.  0.  0.  0.  0.  0.5 0.5]
Resíduos por marginal:
  Marginal AB: Erro Frobenius = 4.57e-15, Max Abs = 2.82e-15
  Marginal AC: Erro Frobenius = 4.57e-15, Max Abs = 2.82e-15
  Marginal BC: Erro Frobenius = 4.57e-15, Max Abs = 2.82e-15


## 3. Exemplo 2: Marginais Maximamente Mistas

### Descrição Física:
Prescrevemos que todas as três marginais de 2 qubits sejam a matriz identidade normalizada (estado de entropia máxima):

$$\omega_{AB} = \omega_{AC} = \omega_{BC} = \frac{I_4}{4}$$

### Regime de Representabilidade Esperado:
- **Representabilidade Mista**: **FACTÍVEL**. O estado global maximamente misto $\rho_{ABC} = \frac{I_8}{8}$ satisfaz exatamente $\text{Tr}_C(I_8 / 8) = I_4 / 4$.
- **Representabilidade Pura**: **INFACTÍVEL**. Nenhum estado puro de 3 qubits pode possuir todas as três marginais bipartidas maximamente mistas.


In [4]:
# Montagem das marginais maximamente mistas
omega_max_mixed = np.eye(4, dtype=complex) / 4.0

marginals_ex2 = {
    "AB": omega_max_mixed,
    "AC": omega_max_mixed,
    "BC": omega_max_mixed,
}

print("Exemplo 2 (Maximamente Misto)")
sdp_ex2, M_ex2, b_ex2 = build_qmp_sdp(marginals_ex2, dims=[2, 2, 2])
print(f"SDP construído: dimensão do domínio = {sdp_ex2.dom.n}x{sdp_ex2.dom.n}, restrições m = {len(b_ex2)}")

# Resolução com CLARABEL
res_ex2 = solve_qmp(sdp_ex2, solver="CLARABEL")
print(f"Status do Solver: {res_ex2['status']} | Factível: {res_ex2['is_feasible']}")

# Auditoria
verif_ex2 = verify_marginals(res_ex2["rho"], marginals_ex2, dims=[2, 2, 2])
print(f"Erro de Hermiticidade:  {verif_ex2['hermitian_error']:.2e}")
print(f"Traço Tr(rho):          {verif_ex2['trace'].real:.12f} (Erro: {verif_ex2['trace_error']:.2e})")
print(f"lambda_min:             {verif_ex2['min_eigenvalue']:.2e} -> rho >= 0: {verif_ex2['is_psd']}")
print(f"Pureza Tr(rho^2):       {verif_ex2['purity']:.4f} (Esperado para I_8/8: 0.1250)")
print("Autovalores de rho_ABC:")
print(" ", np.round(verif_ex2["eigenvalues"], 6))
print("Resíduos por marginal:")
for k, err in verif_ex2["marginal_errors"].items():
    print(f"  Marginal {k}: Erro Frobenius = {err['frobenius_error']:.2e}, Max Abs = {err['max_abs_error']:.2e}")

Exemplo 2 (Maximamente Misto)
SDP construído: dimensão do domínio = 8x8, restrições m = 49
Status do Solver: optimal | Factível: True
Erro de Hermiticidade:  0.00e+00
Traço Tr(rho):          1.000000000000 (Erro: 0.00e+00)
lambda_min:             1.25e-01 -> rho >= 0: True
Pureza Tr(rho^2):       0.1250 (Esperado para I_8/8: 0.1250)
Autovalores de rho_ABC:
  [0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]
Resíduos por marginal:
  Marginal AB: Erro Frobenius = 0.00e+00, Max Abs = 0.00e+00
  Marginal AC: Erro Frobenius = 0.00e+00, Max Abs = 0.00e+00
  Marginal BC: Erro Frobenius = 0.00e+00, Max Abs = 0.00e+00


## 4. Exemplo 3: Incompatibilidade de Bell e Monogamia do Emaranhamento

### Descrição Física:
Exigimos que o qubit $A$ esteja **simultaneamente** maximamente emaranhado em um estado puro de Bell $|\Phi^+\rangle$ com o qubit $B$ e com o qubit $C$:

$$\omega_{AB} = |\Phi^+\rangle\langle\Phi^+|, \quad \omega_{AC} = |\Phi^+\rangle\langle\Phi^+|$$

### Teste de Sobreposição vs Factibilidade SDP:
1. **Condição de Sobreposição**: É satisfeita, pois $\text{Tr}_B(\omega_{AB}) = \frac{I_2}{2} = \text{Tr}_C(\omega_{AC})$.
2. **Representabilidade Mista/Pura**: **INFACTÍVEL**. A Teoria Quantum proíbe que um sistema esteja maximalmente emaranhado com dois sistemas distintos simultaneamente (**Monogamia do Emaranhamento**). O solver **CLARABEL** retorna status infactível.


In [ ]:
# Montagem das marginais incompatíveis de Bell
phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2.0)
rho_bell = ketbra(phi_plus)

marginals_ex3 = {
    "AB": rho_bell,
    "AC": rho_bell,
}

#Vamos fazer um tete de consistência

rho_A_de_AB = partial_trace(marginals_ex3["AB"], keep=[0], dims=[2,2])
rho_A_de_AC = partial_trace(marginals_ex3["AC"], keep=[0], dims=[2,2])

overlap_error = np.linalg.norm(
    rho_A_de_AB - rho_A_de_AC,
    ord="fro"
)

print("\nTeste de consistência na sobreposição:")
print("Marginal de A obtida a partir de AB:")
print(rho_A_de_AB)

print("\nMarginal de A obtida a partir de AC:")
print(rho_A_de_AC)

print(f"\nErro de sobreposição ||rho_A^(AB) - rho_A^(AC)||_F = {overlap_error:.3e}")

if np.isclose(overlap_error, 0.0):
    print("A condição de sobreposição é satisfeita.")
    print("Entretanto, isso é apenas uma condição necessária, não suficiente,")
    print("para a existência de uma extensão global.")
else:
    print("A condição de sobreposição é violada.")


print("Exemplo 3 (Incompatibilidade de Bell)")
sdp_ex3, M_ex3, b_ex3 = build_qmp_sdp(marginals_ex3, dims=[2, 2, 2])
print(f"SDP construído: dimensão do domínio = {sdp_ex3.dom.n}x{sdp_ex3.dom.n}, restrições m = {len(b_ex3)}")

# Resolução com CLARABEL
res_ex3 = solve_qmp(sdp_ex3, solver="CLARABEL")
print(f"Status retornado pelo solver: {res_ex3['status']}")
print(f"O problema é factível?        {res_ex3['is_feasible']}")
print(f"Diagnóstico capturado:        {res_ex3['error_message']}")



Teste de consistência na sobreposição:
Marginal de A obtida a partir de AB:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

Marginal de A obtida a partir de AC:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

Erro de sobreposição ||rho_A^(AB) - rho_A^(AC)||_F = 0.000e+00
A condição de sobreposição é satisfeita.
Entretanto, isso é apenas uma condição necessária, não suficiente,
para a existência de uma extensão global.
Exemplo 3 (Incompatibilidade de Bell)
SDP construído: dimensão do domínio = 8x8, restrições m = 33
Status retornado pelo solver: infeasible
O problema é factível?        False
Diagnóstico capturado:        CLARABEL solver did not return a solution (infeasible).
